In [1]:
import pandas as pd
from fredapi import Fred
fred = Fred(api_key = '621a31e8c3e8dcea20a5c0db2bafb43c')

In [2]:
df = pd.read_csv('term_life_experience_study.csv')

In [38]:
unemployment = pd.read_csv('UNRATE.csv')
sentiment = pd.read_csv('UMCSENT.csv')
income = pd.read_csv('MEHOINUSA672N.csv')
treasury_spread = pd.read_csv('T10Y2Y.csv')

In [35]:
print(df[['Policy_ID', 'Issue_Year', 'Lapse_Year', 'Policy_Status']].head())

   Policy_ID  Issue_Year  Lapse_Year Policy_Status
0  POL100000        2006         NaN        Active
1  POL100001        2007         NaN        Active
2  POL100002        2008         NaN        Active
3  POL100003        2006         NaN         Death
4  POL100004        2009         NaN        Active


In [36]:
print(unemployment.head())

  observation_date  UNRATE
0       1948-01-01     3.4
1       1948-02-01     3.8
2       1948-03-01     4.0
3       1948-04-01     3.9
4       1948-05-01     3.5


In [37]:
print(sentiment.head())

  observation_date  UMCSENT
0       1952-11-01     86.2
1       1952-12-01      NaN
2       1953-01-01      NaN
3       1953-02-01     90.7
4       1953-03-01      NaN


In [7]:
print(income.head())

  observation_date  MEHOINUSA672N
0       1984-01-01          60420
1       1985-01-01          61570
2       1986-01-01          63850
3       1987-01-01          64650
4       1988-01-01          65130


In [39]:
print(treasury_spread.head())

  observation_date  T10Y2Y
0       1976-06-01    0.68
1       1976-06-02    0.71
2       1976-06-03    0.70
3       1976-06-04    0.77
4       1976-06-07    0.79


In [8]:
unemployment['observation_date'] = pd.to_datetime(unemployment['observation_date'])
unemployment['Year'] = unemployment['observation_date'].dt.year
unemployment_yearly = unemployment.groupby('Year')['UNRATE'].mean().reset_index()
unemployment_yearly = unemployment_yearly.rename(columns={'UNRATE': 'Unemployment_Rate'})

In [10]:
sentiment['observation_date'] = pd.to_datetime(sentiment['observation_date'])
sentiment['Year'] = sentiment['observation_date'].dt.year

sentiment_yearly = sentiment.groupby('Year')['UMCSENT'].mean().reset_index()
sentiment_yearly = sentiment_yearly.rename(columns={'UMCSENT': 'Consumer_sentiment'})

In [12]:
income['observation_date'] = pd.to_datetime(income['observation_date'])
income['Year'] = income['observation_date'].dt.year

income_yearly = income.dropna(subset = ['MEHOINUSA672N'])

income_yearly = income_yearly.rename(columns={'MEHOINUSA672N': 'Median_household_income'})
income_yearly = income_yearly[['Year','Median_household_income']]

In [40]:
treasury_spread['observation_date'] = pd.to_datetime(treasury_spread['observation_date'])
treasury_spread['Year'] = treasury_spread['observation_date'].dt.year

treasury_spread_year_ave = treasury_spread.groupby('Year')['T10Y2Y'].mean().reset_index()
treasury_spread_year_ave = treasury_spread_year_ave.rename(columns={'T10Y2Y': 'Average_Treasury_spread'})

In [43]:
macro = unemployment_yearly.merge(sentiment_yearly)
macro = macro.merge(income_yearly)
macro = macro.merge(treasury_spread_year_ave)

In [44]:
macro

,Year,Unemployment_Rate,Consumer_sentiment,Median_household_income,Average_Treasury_spread
0,1984,7.508333,97.475000,60420,0.787831
1,1985,7.191667,93.166667,61570,1.352097
2,1986,7.000000,94.791667,63850,0.812360
3,1987,6.175000,90.641667,64650,0.977040
4,1988,5.491667,93.733333,65130,0.747600
5,1989,5.258333,92.758333,66240,-0.078680
6,1990,5.616667,81.616667,65440,0.390880
7,1991,6.850000,77.550000,63530,1.374600
8,1992,7.491667,77.250000,63010,2.238964
9,1993,6.908333,82.791667,62700,1.820480


In [45]:
df_merged = df.merge(macro, left_on = 'Issue_Year', right_on = 'Year', how = 'left')

In [46]:
df_merged

,Policy_ID,Issue_Year,Term_Length,Birth_Year,Gender,Smoker_Status,Face_Amount,Annual_Premium,Post_Term_Premium,Issued_Age,...,BMI,ZIP_Code,Number_of_Beneficiaries,Mortality_Rate,Lapse_Rate,Year,Unemployment_Rate,Consumer_sentiment,Median_household_income,Average_Treasury_spread
0,POL100000,2006,20.0,1958,F,Non-Smoker,304000,130.72,NaN,48,...,20.4,77005,3,0.001100,0.04,2006.0,4.608333,87.308333,71850.0,-0.023480
1,POL100001,2007,20.0,1944,M,Smoker,252000,146.16,NaN,63,...,23.9,60614,3,0.010700,0.04,2007.0,4.616667,85.583333,73010.0,0.269841
2,POL100002,2008,20.0,1986,M,Non-Smoker,292000,73.00,NaN,0,...,26.9,44101,2,0.001000,0.04,2008.0,5.800000,63.750000,70520.0,1.652948
3,POL100003,2006,20.0,1943,M,Non-Smoker,197000,114.26,NaN,63,...,25.6,2139,2,0.005685,0.04,2006.0,4.608333,87.308333,71850.0,-0.023480
4,POL100004,2009,20.0,1954,F,Non-Smoker,324000,162.00,NaN,55,...,23.9,77005,2,0.001800,0.04,2009.0,9.283333,66.258333,70070.0,2.305640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,POL100995,2003,20.0,1945,F,Non-Smoker,212000,112.36,NaN,58,...,26.4,90210,4,0.002200,0.07,2003.0,5.991667,87.625000,70080.0,2.363280
996,POL100996,2009,20.0,1956,M,Non-Smoker,396000,190.08,NaN,53,...,25.7,90210,2,0.002500,0.04,2009.0,9.283333,66.258333,70070.0,2.305640
997,POL100997,2009,20.0,1988,F,Non-Smoker,234000,58.50,NaN,21,...,18.8,77005,3,0.000300,0.04,2009.0,9.283333,66.258333,70070.0,2.305640
998,POL100998,2002,20.0,1933,M,Non-Smoker,37000,23.68,NaN,69,...,26.1,60614,1,0.007900,0.05,2002.0,5.783333,89.583333,70040.0,1.975320


In [47]:
df_merged.shape

(1000, 27)

In [48]:
macro.shape

(41, 5)